# maxpool-reduce — worked example 2: Max vs avg pool on one window geometry

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `maxpool-reduce`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The same factored-axis reduce pattern builds either pooling type — only the reducer string changes: `'max'` for MaxPool2d, `'mean'` for AvgPool2d. Both give `(..., H/p, W/p)`. They diverge on inputs with outliers, which max preserves and mean dilutes.

## Worked solution

We compute both pools over identical geometry and show where they differ.

1. **Max pool.** `reduce(x, '... (h p1) (w p2) -> ... h w', 'max', p1=p, p2=p)`.
2. **Avg pool.** Same pattern, reducer `'mean'`.
3. **Same shape.** Both outputs are `(..., H/p, W/p)` — the window geometry is shared, only the aggregation differs.
4. **Divergence.** We craft a window with a single large outlier: max keeps it, mean averages it down. Printing both makes the contrast concrete.

The demo pools a tensor with a planted spike and prints the max and mean results for the affected window.

In [ ]:
import torch as t
import einops

t.manual_seed(1)

def max_and_avg(x, p):
    mx = einops.reduce(x, '... (h p1) (w p2) -> ... h w', 'max', p1=p, p2=p)
    av = einops.reduce(x, '... (h p1) (w p2) -> ... h w', 'mean', p1=p, p2=p)
    return mx, av

x = t.zeros(1, 1, 2, 2)
x[0, 0, 0, 0] = 8.0          # single spike in the only 2x2 window
mx, av = max_and_avg(x, 2)
print('max keeps spike:', mx.item())
print('avg dilutes spike:', av.item())   # 8/4 = 2.0
print('same shape:', mx.shape == av.shape)